In [1]:
import pandas as pd
url = "../data/raw/books.csv"
# df = pd.read_csv(url)
# ParserError: Error tokenizing data. C error: Expected 12 fields in line 3350, saw 13

In [2]:
#showing the table without the faulty line
df_preview = pd.read_csv(url, dtype=str, nrows=3348)

In [3]:
df_preview.columns

Index(['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13',
       'language_code', '  num_pages', 'ratings_count', 'text_reviews_count',
       'publication_date', 'publisher'],
      dtype='str')

In [4]:
df_preview.tail()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
3343,12210,Collected Poems Prose and Plays,Robert Frost/Richard Poirier/Mark Richardson,4.32,188301106X,9781883011062,eng,1036,6188,51,10/1/1995,Library of America
3344,12212,Robert Frost: A Life,Jay Parini,4.11,0805063412,9780805063417,eng,514,259,31,3/15/2000,Owl Books (NY)
3345,12216,Three Plays: Desire Under the Elms / Strange I...,Eugene O'Neill/Luann Walther,4.08,0679763961,9780679763963,eng,424,1596,36,10/31/1995,Vintage
3346,12220,A Streetcar Named Desire,Tennessee Williams,3.98,0822210894,9780822210894,eng,107,235224,2409,12/1/1952,Dramatists Play Service
3347,12222,A Streetcar Named Desire,Tennessee Williams/Arthur Miller,3.98,0811216020,9780811216029,en-US,192,3458,337,9/17/2004,New Directions


In [5]:
#showing only the faulty line to understand the issue
problem_row = pd.read_csv(url, skiprows=3349, header=None, nrows=1, dtype=str)

problem_row.T #.T shows the line vertically

# The CSV could not be loaded because line 3350 contained 13 fields instead of 12.
# After inspecting this row, we found that a comma within the author name split the authors field into two columns.

,0
0,12224
1,Streetcar Suburbs: The Process of Growth in Bo...
2,Sam Bass Warner
3,Jr./Sam B. Warner
4,3.58
5,0674842111
6,9780674842113
7,en-US
8,236
9,61


In [6]:
#function to fix the issue with line 3350
def fix_author(row):
    author = row[2] + "," + row[3]
    corrected_row = row[:2] + [author] + row[4:]
    return corrected_row

#checking if the function works as intended
example = problem_row.iloc[0].tolist()
print("Row before fixing:")
print(example)
print("length:", len(example))

fixed_example = fix_author(example)
print("\nRow after fixing:")
print(fixed_example)
print("length:", len(fixed_example))

Row before fixing:
['12224', 'Streetcar Suburbs: The Process of Growth in Boston  1870-1900', 'Sam Bass Warner', ' Jr./Sam B. Warner', '3.58', '0674842111', '9780674842113', 'en-US', '236', '61', '6', '4/20/2004', 'Harvard University Press']
length: 13

Row after fixing:
['12224', 'Streetcar Suburbs: The Process of Growth in Boston  1870-1900', 'Sam Bass Warner, Jr./Sam B. Warner', '3.58', '0674842111', '9780674842113', 'en-US', '236', '61', '6', '4/20/2004', 'Harvard University Press']
length: 12


In [7]:
#Collect rows with extra fields
def collect_bad_rows(row):
    bad_rows.append(row)
    return None

In [8]:
# Reset the list before collecting to avoid duplicate entries
bad_rows = []

df_check = pd.read_csv(url, dtype=str, engine="python", on_bad_lines=collect_bad_rows)
print("Rows with extra fields:", len(bad_rows))

Rows with extra fields: 4


In [9]:
pd.DataFrame(bad_rows).T

,0,1,2,3
0,12224,16914,22128,34889
1,Streetcar Suburbs: The Process of Growth in Bo...,The Tolkien Fan's Medieval Reader,Patriots (The Coming Collapse),Brown's Star Atlas: Showing All The Bright Sta...
2,Sam Bass Warner,David E. Smith (Turgon of TheOneRing.net,James Wesley,Brown
3,Jr./Sam B. Warner,one of the founding members of this Tolkien w...,Rawles,Son & Ferguson
4,3.58,3.58,3.63,0.00
5,0674842111,1593600119,156384155X,0851742718
6,9780674842113,9781593600112,9781563841552,9780851742717
7,en-US,eng,eng,eng
8,236,400,342,49
9,61,26,38,0


In [10]:
# Correct each malformed row using our tested function
fixed_rows = []

for row in bad_rows:
    fixed_rows.append(fix_author(row))

fixed_books = pd.DataFrame(fixed_rows,columns=df_check.columns)

df_clean = pd.concat([df_check, fixed_books], ignore_index=True)

print("Before adding corrected rows:", df_check.shape)
print("After adding corrected rows:", df_clean.shape)

Before adding corrected rows: (11119, 12)
After adding corrected rows: (11123, 12)


In [11]:
#Starting to clean the columns
df_clean.columns.tolist()

['bookID',
 'title',
 'authors',
 'average_rating',
 'isbn',
 'isbn13',
 'language_code',
 '  num_pages',
 'ratings_count',
 'text_reviews_count',
 'publication_date',
 'publisher']

In [12]:
#Removing blank spaces
df_clean.columns = df_clean.columns.str.strip()
df_clean.columns.tolist()

['bookID',
 'title',
 'authors',
 'average_rating',
 'isbn',
 'isbn13',
 'language_code',
 'num_pages',
 'ratings_count',
 'text_reviews_count',
 'publication_date',
 'publisher']

In [13]:
# Count missing values in each column
df_clean.isna().sum()
# No missing values were detected in the loaded data.

bookID                0
title                 0
authors               0
average_rating        0
isbn                  0
isbn13                0
language_code         0
num_pages             0
ratings_count         0
text_reviews_count    0
publication_date      0
publisher             0
dtype: int64

In [14]:
print("Duplicated rows? -", df_clean.duplicated().sum())
# No fully duplicated rows were found.
print("\nDuplicated book IDs? -", df_clean["bookID"].duplicated().sum())
# No duplicate book IDs were found.

Duplicated rows? - 0

Duplicated book IDs? - 0


In [15]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 11123 entries, 0 to 11122
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   bookID              11123 non-null  str  
 1   title               11123 non-null  str  
 2   authors             11123 non-null  str  
 3   average_rating      11123 non-null  str  
 4   isbn                11123 non-null  str  
 5   isbn13              11123 non-null  str  
 6   language_code       11123 non-null  str  
 7   num_pages           11123 non-null  str  
 8   ratings_count       11123 non-null  str  
 9   text_reviews_count  11123 non-null  str  
 10  publication_date    11123 non-null  str  
 11  publisher           11123 non-null  str  
dtypes: str(12)
memory usage: 2.4 MB


In [16]:
# Convert whole-number columns to integers
df_clean["bookID"] = df_clean["bookID"].astype(int)
df_clean["num_pages"] = df_clean["num_pages"].astype(int)
df_clean["ratings_count"] = df_clean["ratings_count"].astype(int)
df_clean["text_reviews_count"] = df_clean["text_reviews_count"].astype(int)

# Ratings can contain decimal places
df_clean["average_rating"] = df_clean["average_rating"].astype(float)

In [17]:
df_clean.dtypes

bookID                  int64
title                     str
authors                   str
average_rating        float64
isbn                      str
isbn13                    str
language_code             str
num_pages               int64
ratings_count           int64
text_reviews_count      int64
publication_date          str
publisher                 str
dtype: object

In [18]:
df_clean[["average_rating", "num_pages", "ratings_count", "text_reviews_count"]].describe().round(2)

,average_rating,num_pages,ratings_count,text_reviews_count
count,11123.00,11123.00,11123.00,11123.00
mean,3.93,336.41,17941.88,541.97
std,0.35,241.15,112499.27,2576.63
min,0.00,0.00,0.00,0.00
25%,3.77,192.00,104.00,9.00
50%,3.96,299.00,745.00,46.00
75%,4.13,416.00,4993.50,237.50
max,5.00,6576.00,4597666.00,94265.00


In [19]:
#Checking how many zero values there are
print("Zero page counts:", (df_clean["num_pages"] == 0).sum())
print("Zero average ratings:", (df_clean["average_rating"] == 0).sum())
print("Zero rating counts:", (df_clean["ratings_count"] == 0).sum())
print("Zero review counts:", (df_clean["text_reviews_count"] == 0).sum())

Zero page counts: 76
Zero average ratings: 26
Zero rating counts: 81
Zero review counts: 625


In [20]:
# Inspect books with a page count of zero, showing only a few columns
df_clean[df_clean["num_pages"] == 0][["bookID", "title", "authors", "num_pages", "publisher"]].head(10)

,bookID,title,authors,num_pages,publisher
306,955,The 5 Love Languages / The 5 Love Languages Jo...,Gary Chapman,0,Moody Publishers
853,2835,The Tragedy of Pudd'nhead Wilson,Mark Twain/Michael Prichard,0,Tantor Media
1061,3593,Murder by Moonlight & Other Mysteries (New Adv...,NOT A BOOK,0,Simon Schuster Audio
1064,3599,The Unfortunate Tobacconist & Other Mysteries ...,NOT A BOOK,0,Simon & Schuster Audio
1230,4249,The Da Vinci Code (Robert Langdon #2),Dan Brown/Paul Michael,0,Random House Audio
1558,5362,The Summons / The Brethren,John Grisham/Michael Beck/Frank Muller,0,Random House Audio
1841,6542,The Patricia Cornwell CD Audio Treasury: All T...,Patricia Cornwell/Kate Burton,0,HarperAudio
1932,6953,Like Water for Chocolate,Laura Esquivel/Yareli Arizmendi,0,Random House Audio
2104,7598,A Study Guide to Gabriel Garcia Marquez' One H...,Gabriel García Márquez/Brenda K. Marshall/F. M...,0,Warner Adult
2128,7674,The Michael Crichton Collection: Airframe / Th...,Michael Crichton/Stephen Lang/Anthony Heald/Bl...,0,Random House Audio


In [21]:
# Treat zero pages as unavailable or not applicable, as some of these are audiobooks
df_clean["num_pages"] = df_clean["num_pages"].replace(0, float("nan"))
df_clean["num_pages"].isna().sum()

np.int64(76)

In [22]:
#Checking the remaining page numbers for mistakes
print(df_clean["num_pages"].describe().round(2))
print("\nNegative page counts:", (df_clean["num_pages"] < 0).sum())

count    11047.00
mean       338.73
std        240.36
min          1.00
25%        196.00
50%        301.00
75%        416.00
max       6576.00
Name: num_pages, dtype: float64

Negative page counts: 0


In [23]:
#Checking the largest page counts
df_clean[["title", "num_pages", "publisher"]].sort_values("num_pages", ascending=False).head(10)

# Large page counts include collections, boxed sets and reference books.
# Keeping these values because size alone does not indicate an error.

,title,num_pages,publisher
6495,The Complete Aubrey/Maturin Novels (5 Volumes),6576.0,W. W. Norton Company
6800,The Second World War,4736.0,Mariner Books
10902,Remembrance of Things Past (Boxed Set),3400.0,Chatto & Windus
6,Harry Potter Collection (Harry Potter #1-6),3342.0,Scholastic
6820,Summa Theologica 5 Vols,3020.0,Christian Classics
7860,Harrison's Principles of Internal Medicine,2751.0,McGraw-Hill Medical Publishing
4,Harry Potter Boxed Set Books 1-5 (Harry Potte...,2690.0,Scholastic
10746,The Sword of Truth Boxed Set I: Wizard's Firs...,2480.0,Tor Books
6255,The J.R.R. Tolkien Companion and Guide,2264.0,Houghton Mifflin Harcourt
5612,Study Bible: NIV,2198.0,Zondervan Publishing House


In [24]:
#Checking the smallest page counts
df_clean[["title", "num_pages", "publisher"]].sort_values("num_pages").head(20)

,title,num_pages,publisher
5648,Meditations for Manifesting: Morning and Eveni...,1.0,Hay House
3453,Essential Welty: Why I Live at the P.O. A Mem...,1.0,Caedmon
10881,Matilda,1.0,Puffin Audiobooks
4942,Lobsters,1.0,AudioText
4943,Antibodies,1.0,AudioText
4269,The Return of the King (The Lord of the Rings ...,1.0,BBC Audiobooks Ltd
9034,There Was a Cold Lady Who Swallowed Some Snow!,1.0,Scholastic Audio Books
5777,The Best American Mystery Stories 2002,1.0,Mariner Books
5753,The Best American Mystery Stories 2003,1.0,Mariner Books
5942,The Neil Gaiman Audio Collection,1.0,HarperFestival


In [25]:
# Inspect unusually small page counts
small_page_books = df_clean[df_clean["num_pages"] <= 2]

print("Books with 1 page:", (df_clean["num_pages"] == 1).sum())
print("Books with 2 pages:", (df_clean["num_pages"] == 2).sum())

small_page_books[["bookID", "title", "num_pages", "publisher"]]

#Several records with 1-2 pages appear to be audiobooks

Books with 1 page: 11
Books with 2 pages: 15


,bookID,title,num_pages,publisher
370,1281,Men Are from Mars Women Are from Venus,2.0,HarperAudio
636,2059,The Long Goodbye (Philip Marlowe #6),2.0,BBC Worldwide
2849,10581,The Mist,2.0,Simon & Schuster Audio
3453,12582,Essential Welty: Why I Live at the P.O. A Mem...,1.0,Caedmon
4269,15330,The Return of the King (The Lord of the Rings ...,1.0,BBC Audiobooks Ltd
4542,16357,Hercule Poirot's Christmas: A BBC Radio 4 Full...,2.0,BBC Audiobooks
4561,16414,Joy in the Morning,2.0,BBC Audiobooks
4942,17872,Lobsters,1.0,AudioText
4943,17873,Antibodies,1.0,AudioText
5472,19786,The Goon Show Volume 4: My Knees Have Fallen ...,2.0,BBC Physical Audio


In [26]:
#Checking zero value ratings
df_clean[df_clean["average_rating"] == 0][["bookID", "title", "average_rating", "ratings_count"]]

,bookID,title,average_rating,ratings_count
265,799,Out to Eat London 2002 (Lonely Planet Out to Eat),0.0,0
375,1302,Juiced Official Strategy Guide,0.0,0
987,3351,Open City 6: The Only Woman He Ever Left,0.0,0
2531,9337,How To Meditate: An Anthology Of Talks On Medi...,0.0,0
2532,9338,Insights: Talks On The Nature Of Existence,0.0,0
2757,10200,Venac sonetnih venaca; Puževa srma,0.0,0
3491,12712,Brodie's notes on Aldous Huxley's brave new world,0.0,0
4240,15186,American Film Guide,0.0,0
4675,16806,The Man and the Author: John Milton: Twentieth...,0.0,0
5321,19257,Canopy: A Work for Voice and Light in Harvard ...,0.0,0


In [27]:
#All books with 0 rating average also have 0 ratings, which means it shouldn't count for the average.
df_clean["average_rating"] = df_clean["average_rating"].replace(0, float("nan"))
df_clean["average_rating"].isna().sum()

np.int64(26)

In [28]:
# Check for negative rating counts
(df_clean["ratings_count"] < 0).sum()

np.int64(0)

In [29]:
# Count records with negative review counts
(df_clean["text_reviews_count"] < 0).sum()

np.int64(0)

In [30]:
# Count how many records use each language code
df_clean["language_code"].value_counts()

language_code
eng      8909
en-US    1407
spa       218
en-GB     214
fre       144
ger        99
jpn        46
mul        19
zho        14
grc        11
por        10
en-CA       7
ita         5
enm         3
lat         3
rus         2
swe         2
ara         1
nl          1
srp         1
msa         1
glg         1
wel         1
nor         1
tur         1
gla         1
ale         1
Name: count, dtype: int64

In [31]:
#Put together all the english codes
english_codes = {
    "en-US": "eng",
    "en-GB": "eng",
    "en-CA": "eng"
}

df_clean["language_code"] = df_clean["language_code"].replace(english_codes)
df_clean["language_code"].value_counts()

language_code
eng    10537
spa      218
fre      144
ger       99
jpn       46
mul       19
zho       14
grc       11
por       10
ita        5
enm        3
lat        3
rus        2
swe        2
ara        1
nl         1
srp        1
msa        1
glg        1
wel        1
nor        1
tur        1
gla        1
ale        1
Name: count, dtype: int64

In [32]:
#Checking the 20 most frequent publishers
print("Total books by the top most 20 publishers:", df_clean["publisher"].value_counts().head(20).sum())

df_clean["publisher"].value_counts().head(20)

Total books by the top most 20 publishers: 2364


publisher
Vintage                         318
Penguin Books                   261
Penguin Classics                184
Mariner Books                   150
Ballantine Books                144
Harper Perennial                112
HarperCollins                   112
Pocket Books                    111
Bantam                          110
VIZ Media LLC                    88
Berkley                          86
Dover Publications               85
Modern Library                   82
Del Rey                          80
Tor Books                        76
Grand Central Publishing         76
Oxford University Press  USA     75
Scribner                         73
Oxford University Press          73
W. W. Norton  Company            68
Name: count, dtype: int64

In [33]:
#Checking for the most common publishers with incorrect spelling
for publisher in sorted(df_clean["publisher"].unique()):
    name = publisher.lower()
    if "harp" in name or "pengu" in name or "oxford" in name or "vintage" in name or "mariner" in name or "pocket" in name or "bantam" in name or "viz" in name or "berkley" in name or "dover" in name or "modern" in name or "del rey" in name or "tor books" in name or "grand" in name or "scribn" in name or "norton" in name:
        print(repr(publisher))

'Arkana/Penguin'
'Atlas Books/HarperCollins Publishers'
'Avon HarperCollins'
'Bantam'
'Bantam (Fanfare Imprint)'
'Bantam (Fanfare)'
'Bantam Books'
'Bantam Books  Inc.'
'Bantam Books (NY)'
'Bantam Classic'
'Bantam Classics'
'Bantam Doubleday Dell (NYC)'
'Bantam Doubleday Dell Publishing Company Inc.'
'Bantam Doubleday Dell Publishing Group'
'Bantam Press'
'Bantam Skylark'
'Bantam Spectra'
'Bantam Spectra (NY)'
'Bantam Starfire'
'BantamSpectra'
'Basil Blackwell (Cambridge  MA/Oxford)'
'Berkley'
'Berkley Books'
'Berkley Books (NY)'
'Berkley Hardcover'
'Berkley Heat'
'Berkley Prime Crime'
'Berkley Prime Crime Books'
'Berkley Publishing'
'Berkley Publishing Group'
'Berkley Sensation'
'Berkley Trade'
"Charles Scribner's Sons"
'Collins Modern Classics'
'Del Rey'
'Del Rey Ballantine Random House'
'Del Rey Books'
'Del Rey Books (NY)'
'Del Rey Fantasy'
'Del Rey/Ballantine Books'
'Dover Publications'
'Dover Publications  Inc.'
'Dover Publications (NY)'
'Dover Publications (NYC)'
'Ecco / HarperCol

In [34]:
#Correct publisher names
publisher_corrections = {
    # Bantam
    "Bantam (Fanfare Imprint)": "Bantam",
    "Bantam (Fanfare)": "Bantam",
    "Bantam Books": "Bantam",
    "Bantam Books  Inc.": "Bantam",
    "Bantam Books (NY)": "Bantam",
    "Bantam Classic": "Bantam",
    "Bantam Classics": "Bantam",
    "Bantam Doubleday Dell (NYC)": "Bantam",
    "Bantam Doubleday Dell Publishing Company Inc.": "Bantam",
    "Bantam Doubleday Dell Publishing Group": "Bantam",
    "Bantam Press": "Bantam",
    "Bantam Skylark": "Bantam",
    "Bantam Spectra": "Bantam",
    "Bantam Spectra (NY)": "Bantam",
    "Bantam Starfire": "Bantam",
    "BantamSpectra": "Bantam",
    'Spectra/Bantam Books (NYC)': "Bantam",

    # Berkley
    "Berkley Books": "Berkley",
    "Berkley Books (NY)": "Berkley",
    "Berkley Hardcover": "Berkley",
    "Berkley Heat": "Berkley",
    "Berkley Prime Crime": "Berkley",
    "Berkley Prime Crime Books": "Berkley",
    "Berkley Publishing": "Berkley",
    "Berkley Publishing Group": "Berkley",
    "Berkley Sensation": "Berkley",
    "Berkley Trade": "Berkley",
    'The Berkley Publishing Group': "Berkley",
    'Underwood/Miller  Berkley': "Berkley",

    # Del Rey
    "Del Rey Ballantine Random House": "Del Rey",
    "Del Rey Books": "Del Rey",
    "Del Rey Books (NY)": "Del Rey",
    "Del Rey Fantasy": "Del Rey",
    "Del Rey/Ballantine Books": "Del Rey",

    # Dover Publications
    "Dover Publications  Inc.": "Dover Publications",
    "Dover Publications (NY)": "Dover Publications",
    "Dover Publications (NYC)": "Dover Publications",

    # Grand Central Publishing
    "Grand Central": "Grand Central Publishing",
    "Grand Central Life & Style": "Grand Central Publishing",
    "Grand Central Publishing (Formerly Warner Books)": "Grand Central Publishing",

    # HarperCollins
    "Atlas Books/HarperCollins Publishers": "HarperCollins",
    "Avon HarperCollins": "HarperCollins",
    "Ecco / HarperCollins": "HarperCollins",
    "Ecco/Harper Perennial": "HarperCollins",
    "Harper": "HarperCollins",
    "Harper & Row": "HarperCollins",
    "Harper & Row (NYC et al.)": "HarperCollins",
    "Harper Business": "HarperCollins",
    "Harper Design": "HarperCollins",
    "Harper Element": "HarperCollins",
    "Harper Paperbacks": "HarperCollins",
    "Harper Perennial (NYC)": "HarperCollins",
    "Harper Perennial Classics": "HarperCollins",
    "Harper Perennial Modern Classics": "HarperCollins",
    "Harper Teen": "HarperCollins",
    "Harper Trophy": "HarperCollins",
    "Harper Voyager": "HarperCollins",
    "HarperCollins (SanFrancisco)": "HarperCollins",
    "HarperCollins Espanol": "HarperCollins",
    "HarperCollins Leadership": "HarperCollins",
    "HarperCollins Publishers Inc.": "HarperCollins",
    "HarperCollins Publishers Ltd": "HarperCollins",
    "HarperCollins UK": "HarperCollins",
    "HarperCollinsChildren’sBooks": "HarperCollins",
    "HarperEntertainment": "HarperCollins",
    "HarperFestival": "HarperCollins",
    "HarperLargePrint": "HarperCollins",
    "HarperOne": "HarperCollins",
    "HarperPaperbacks": "HarperCollins",
    "HarperPerennial / Perennial Classics": "HarperCollins",
    "HarperPrism": "HarperCollins",
    "HarperResource": "HarperCollins",
    "HarperSanFrancisco": "HarperCollins",
    "HarperTeen": "HarperCollins",
    "HarperTempest": "HarperCollins",
    "HarperTorch": "HarperCollins",
    "HarperTorch (an imprint of HarperCollins Publishers)": "HarperCollins",
    "HarperTrophy": "HarperCollins",
    "HarperVoyager": "HarperCollins",
    "Harpercollins Children's Books": "HarperCollins",
    "Harpercollins Childrens Books": "HarperCollins",
    "Harpperen": "HarperCollins",
    "Joanna Cotler Books/HarperCollinsPublishers": "HarperCollins",
    "Perennial / William Morrow / HarperCollins": "HarperCollins",
    "PerfectBound (HarperCollins)": "HarperCollins",
    "William Morrow / Harper": "HarperCollins",
    "William Morrow / HarperCollins / Harper Perennial": "HarperCollins",
    'Collins Modern Classics': "HarperCollins",

    # Modern Library
    'Modern Library (NY)': 'Modern Library',
    'Modern Library Classics': 'Modern Library',
    'Modern Library/Random House (NY)': 'Modern Library',

    # Oxford University Press
    "Oxford University Press  USA": "Oxford University Press",
    'OUP Oxford': "Oxford University Press",

    # Penguin Books
    "Penguin Classics": "Penguin Books",
    "Penguin Books (Penguin Classics)": "Penguin Books",
    "Penguin Books /Penguin Classics": "Penguin Books",
    'Penguin': "Penguin Books",
    'Penguin (Business)': "Penguin Books",
    'Penguin (Non-Classics)': "Penguin Books",
    'Penguin Book Limited': "Penguin Books",
    'Penguin Books  Limited (UK)': "Penguin Books",
    'Penguin Books (London)': "Penguin Books",
    'Penguin Books (NY)': "Penguin Books",
    'Penguin Books (NYC)': "Penguin Books",
    'Penguin Books Australia Ltd.': "Penguin Books",
    'Penguin Books Canada': "Penguin Books",
    'Penguin Books Limited': "Penguin Books",
    'Penguin Books Ltd': "Penguin Books",
    'Penguin Books Ltd.': "Penguin Books",
    'Penguin Books Ltd. (London)': "Penguin Books",
    'Penguin Books USA Inc.': "Penguin Books",
    'Penguin Books in association with Faber & Faber': "Penguin Books",
    'Penguin Books/Penguin Popular Classics': "Penguin Books",
    'Penguin Classics Deluxe Edition': "Penguin Books",
    'Penguin Classics Deluxe Editions': "Penguin Books",
    'Penguin Essential Editions': "Penguin Books",
    'Penguin Global': "Penguin Books",
    'Penguin Group': "Penguin Books",
    'Penguin Group (USA)': "Penguin Books",
    'Penguin Group(CA)': "Penguin Books",
    'Penguin Press': "Penguin Books",
    'Penguin Press HC  The': "Penguin Books",
    'Penguin Putnam': "Penguin Books",
    'Penguin Signet': "Penguin Books",
    'Penguin UK': "Penguin Books",
    'Penguin Workshop': "Penguin Books",
    'Penguin Young Readers': "Penguin Books",
    'Penguin-HighBridge': "Penguin Books",
    'Penguin/Viking Compass (Non-Classics)': "Penguin Books",
    'Plume (Penguin Books Ltd)': "Penguin Books",
    'New York : Penguin Books': "Penguin Books",
    'Michael Joseph Ltd./Penguin Books Ltd.': "Penguin Books",
    'Puffin Books (Penguin Books)': "Penguin Books",
    'Arkana/Penguin': "Penguin Books",
    'Faber and Faber & Penguin India': "Penguin Books",

    # Pocket Books
    'Pocket': 'Pocket Books',
    'Pocket Books / Simon & Schuster  Inc.': 'Pocket Books',
    'Pocket Books/Simon & Schuster (NY)': 'Pocket Books',
    'Pocket Books/Star Trek': 'Pocket Books',
    'Pocket Books: Gallery Books': 'Pocket Books',
    'Pocket Star': 'Pocket Books',
    'Pocket Star Books': 'Pocket Books',

    # Scribner
    'Scribner Book Company': 'Scribner',
    'Scribner Classics': 'Scribner',
    "Charles Scribner's Sons": 'Scribner',
    'Prentice Hall/Scribner/Macmillan': 'Scribner',

    # The Modern Library
    'The Modern Library New York': 'The Modern Library',
    'Random House: Modern Library': 'The Modern Library',

    # Random House Vintage
    'Random House (Vintage)': "Random House Vintage",
    'Random House Vintage Books': "Random House Vintage",
    'Vintage': 'Random House Vintage',
    'Vintage Books USA': 'Random House Vintage',
    'Vintage Books a division of Random House': 'Random House Vintage',
    'Vintage Books/Vintage Classics': 'Random House Vintage',
    'Vintage Classics': 'Random House Vintage',
    'Vintage Contemporaries': 'Random House Vintage',
    'Vintage Crime': 'Random House Vintage',
    'Vintage Crime/Black Lizard': 'Random House Vintage',
    'Vintage Espanol': 'Random House Vintage',
    'Vintage International': 'Random House Vintage',
    'Vintage/Ebury': 'Random House Vintage',
    'Vintage/Ebury (A Division of Random House Group)': 'Random House Vintage',
    'Vintage/Random House (NY)': 'Random House Vintage',
    'Vintage/Random House Inc. (NYC)': 'Random House Vintage',

    # VIZ Media
    'Viz Media': 'VIZ Media',
    'VIZ Media LLC': 'VIZ Media',

    # W.W. Norton Company
    'W. W. Norton  Company': 'W.W. Norton Company',
    'W. W. Norton & Co.': 'W.W. Norton Company',
    'W. W. Norton & Company': 'W.W. Norton Company',
    'W. W. Norton and Company': 'W.W. Norton Company',
    'W.W. Norton': 'W.W. Norton Company',
    'W.W. Norton  Company': 'W.W. Norton Company',
    'W.W. Norton & Company': 'W.W. Norton Company',
    'W.W. Norton & Company  Inc. (NY)': 'W.W. Norton Company',
    'W.W. Norton & Company (NY)': 'W.W. Norton Company',
    'W.W. Norton & Company (NY/London)': 'W.W. Norton Company',
    'W.W. Norton & Company (NYC)': 'W.W. Norton Company',
    'Norton': 'W.W. Norton Company',
    'Norton and Co.': 'W.W. Norton Company',
    'Liveright/W.W. Norton & Company  Inc.': 'W.W. Norton Company',
}

df_clean["publisher"] = df_clean["publisher"].replace(publisher_corrections)
print("Corrected total books by the top most 20 publishers:", df_clean["publisher"].value_counts().head(20).sum())
df_clean["publisher"].value_counts().head(20)

#The number of books considered in the top 20 publishers increased from 2364 to 3396, and the top 3 changed from 
# Vintage, Penguin Books, and Penguin Classics to Penguin Books, Random House Vintage and HarperCollins

Corrected total books by the top most 20 publishers: 3396


publisher
Penguin Books                 596
Random House Vintage          402
HarperCollins                 389
Bantam                        201
Berkley                       170
Oxford University Press       152
Mariner Books                 150
Pocket Books                  149
Ballantine Books              144
W.W. Norton Company           141
VIZ Media                     122
Del Rey                       118
Harper Perennial              112
Dover Publications             89
Modern Library                 87
Scribner                       86
Grand Central Publishing       80
Tor Books                      76
Signet                         66
Cambridge University Press     66
Name: count, dtype: int64

In [35]:
#cleaning the title column
df_clean["title"] = df_clean["title"].str.strip()

In [36]:
#Authors
df_clean["authors"] = df_clean["authors"].str.strip()

df_clean[["bookID", "authors"]].head(20)

,bookID,authors
0,1,J.K. Rowling/Mary GrandPré
1,2,J.K. Rowling/Mary GrandPré
2,4,J.K. Rowling
3,5,J.K. Rowling/Mary GrandPré
4,8,J.K. Rowling/Mary GrandPré
5,9,W. Frederick Zimmerman
6,10,J.K. Rowling
7,12,Douglas Adams
8,13,Douglas Adams
9,14,Douglas Adams


In [37]:
#Checking for the most common authors with incorrect spelling
for author in sorted(df_clean["authors"].unique()):
    name = author.lower()
    if "rowling" in name or "douglas adams" in name:
        print(repr(author))

'Douglas Adams'
'Douglas Adams/Christopher Cerf'
'Douglas Adams/Dirk Maggs/Simon  Jones/Geoffrey McGivern/Mark Wing-Davey/Susan  Sheridan/Sandra Dickinson/Stephen  Moore/William Franklyn/Rula Lenska/Sam  Burke'
'Douglas Adams/John Lloyd'
'Douglas Adams/Martin  Freeman'
'Douglas Adams/Stephen Fry'
'J.K. Rowling'
'J.K. Rowling/Gemma Rovira Ortega'
'J.K. Rowling/J.K.羅琳/彭倩文'
'J.K. Rowling/J.K.羅琳/皇冠編譯組/張定綺/彭倩文/趙丕慧/林靜華'
'J.K. Rowling/Mary GrandPré'
'J.K. Rowling/Peter Needham'
'J.K. Rowling/Rufus Beck'
'J.K. Rowling/Sevin Okyay'
'Newt Scamander/J.K. Rowling'


In [38]:
#Correct author names so the top 20 show more 
author_corrections = {
    # Douglas Adams
    'Douglas Adams/Christopher Cerf': 'Douglas Adams',
    'Douglas Adams/Dirk Maggs/Simon  Jones/Geoffrey McGivern/Mark Wing-Davey/Susan  Sheridan/Sandra Dickinson/Stephen  Moore/William Franklyn/Rula Lenska/Sam  Burke': 'Douglas Adams',
    'Douglas Adams/John Lloyd': 'Douglas Adams',
    'Douglas Adams/Martin  Freeman': 'Douglas Adams',
    'Douglas Adams/Stephen Fry': 'Douglas Adams',

    # J.K. Rowling
    'J.K. Rowling/Gemma Rovira Ortega': 'J.K. Rowling',
    'J.K. Rowling/J.K.羅琳/彭倩文': 'J.K. Rowling',
    'J.K. Rowling/J.K.羅琳/皇冠編譯組/張定綺/彭倩文/趙丕慧/林靜華': 'J.K. Rowling',
    'J.K. Rowling/Mary GrandPré': 'J.K. Rowling',
    'J.K. Rowling/Peter Needham': 'J.K. Rowling',
    'J.K. Rowling/Rufus Beck': 'J.K. Rowling',
    'J.K. Rowling/Sevin Okyay': 'J.K. Rowling',
    'Newt Scamander/J.K. Rowling': 'J.K. Rowling',
}

df_clean["authors"] = df_clean["authors"].replace(author_corrections)
df_clean["authors"].value_counts().head(20)

authors
Stephen King                   40
P.G. Wodehouse                 40
Rumiko Takahashi               39
Orson Scott Card               35
Agatha Christie                33
Piers Anthony                  30
Mercedes Lackey                29
Sandra Brown                   29
Dick Francis                   28
J.K. Rowling                   25
Margaret Weis/Tracy Hickman    23
Terry Pratchett                23
James Patterson                23
Laurell K. Hamilton            23
Gordon Korman                  22
Bill Bryson                    21
Janet Evanovich                21
Dan Simmons                    21
Alan Dean Foster               21
Gabriel García Márquez         20
Name: count, dtype: int64

In [39]:
#Checking for the updated most common authors with incorrect spelling
for author in sorted(df_clean["authors"].unique()):
    name = author.lower()
    if "king" in name or "wodehouse" in name or "rumiko" in name or "card" in name or "agatha" in name or "piers" in name or "mercedes" in name or "brown" in name or "francis" in name or "rowling" in name or "weis" in name or "hickman" in name or "pratchett" in name or "patterson" in name or "hamilton" in name or "korman" in name or "bryson" in name or "evanovich" in name or "simmons" in name or "foster" in name or "marquez" in name or "márquez" in name or "marquéz" in name or "tolkien" in name:
        print(repr(author))

'A.E. Cunningham/Harlan Ellison/Charles F. Miller/Terry Dowling/Paul Rhoads/Jack Vance/Tom Shippey/Gene Wolfe/David Langford/Dan Simmons/David Mathew'
'Agatha Christie'
'Agatha Christie/David Suchet'
'Agatha Christie/Hugh Fraser'
'Agatha Christie/Hugh Fraser/David Suchet'
'Agatha Christie/Michael Bakewell/June Whitfield/Nicky Henson/Derek Waring/Natasha Pyne'
'Agatha Christie/Michael Bakewell/Peter Sallis/Cyril Luckham/Edward De Souza/Rachel Gurney/Nicky Henson/Deryck Guyler'
'Agatha Christie/Robert Welch Herrick'
'Agatha Christie/Rosalind Ayres'
'Agatha Christie/Rosemary Leach'
'Alan Dean Foster'
'Alan Dean Foster/George Alec Effinger/Jack McDevitt/Nina Kiriki Hoffman/Mike Resnick/R.A. Lafferty/Tobias Grace/Laura Resnick/Margaret Ball/Ron Goulart/Mel Gilden/Edward Wellen/Wolfgang Jeschke/Greg Costikyan/Esther M. Friesner/Steve Rasnic Tem'
'Alan Dean Foster/James Gurney'
'Allan C. Weisbecker'
'Anne Mazer/Bill Brown'
'Anthony J. Cascardi'
"Arthur Conan Doyle/Francis O'Gorman"
'Augustine

In [40]:
# Correct names keeping colaborations but simplifying for translations
more_author_corrections = {
    # Agatha Christie
    "Agatha Christie/David Suchet": "Agatha Christie",
    "Agatha Christie/Hugh Fraser": "Agatha Christie",
    "Agatha Christie/Hugh Fraser/David Suchet": "Agatha Christie",
    "Agatha Christie/Michael Bakewell/June Whitfield/Nicky Henson/Derek Waring/Natasha Pyne": "Agatha Christie",
    "Agatha Christie/Michael Bakewell/Peter Sallis/Cyril Luckham/Edward De Souza/Rachel Gurney/Nicky Henson/Deryck Guyler": "Agatha Christie",
    "Agatha Christie/Rosalind Ayres": "Agatha Christie",
    "Agatha Christie/Rosemary Leach": "Agatha Christie",
    "Mary Westmacott/Agatha Christie": "Agatha Christie",

    # Bill Bryson
    "Bill  Bryson/Kathryn Lamb": "Bill Bryson",
    "Bill Bryson/David B. Ellis": "Bill Bryson",
    "Bill Bryson/José Manuel Álvarez": "Bill Bryson",

    # Dan Brown
    "Dan Brown/Daniel Roche": "Dan Brown",
    "Dan Brown/Eduardo García Murillo": "Dan Brown",
    "Dan Brown/Juanjo Estrella": "Dan Brown",
    "Dan Brown/Paul Michael": "Dan Brown",
    "Dan Brown/Raúl Amundaray": "Dan Brown",

    # Dan Simmons
    "Dan Simmons/Carlos Alberto Gardini": "Dan Simmons",
    "Dan Simmons/Guy Abadia": "Dan Simmons",
    "Dan Simmons/Jean-Daniel Brèque": "Dan Simmons",
    "Dan Simmons/Joachim Körber": "Dan Simmons",

    # Gabriel García Márquez
    "Gabriel García Márquez/Curt Meyer-Clason": "Gabriel García Márquez",
    "Gabriel García Márquez/Edith Grossman": "Gabriel García Márquez",
    "Gabriel García Márquez/Gregory Rabassa": "Gabriel García Márquez",
    "Gabriel García Márquez/Gregory Rabassa/J.S. Bernstein": "Gabriel García Márquez",
    "Gabriel García Márquez/Randolf Hogan": "Gabriel García Márquez",
    "Gabriel García Márquez/Remy Gorga Filho": "Gabriel García Márquez",

    # J.R.R. Tolkien
    
    "J.R.R. Tolkien/Francis Ledoux": "J.R.R. Tolkien",
    "J.R.R. Tolkien/Alan  Lee": "J.R.R. Tolkien",
    "J.R.R. Tolkien/Rob Inglis": "J.R.R. Tolkien",
    "J.R.R. Tolkien/托爾金/Alan  Lee/朱學恆": "J.R.R. Tolkien",
    
    # Other editions with extra credits
    "Gordon Korman/Holter Graham": "Gordon Korman",
    "James Patterson/مریم کاظمی\u200cتبار": "James Patterson",
    "Janet Evanovich/C.J. Critt": "Janet Evanovich",
    "Janet Evanovich/Lorelei King": "Janet Evanovich",
    "Orson Scott Card/David Birney/Scott Brick/Gabrielle de Cuir/Scott Sowers": "Orson Scott Card",
    "Orson Scott Card/Piotr W. Cholewa": "Orson Scott Card",
    "P.G. Wodehouse/Jonathan Cecil": "P.G. Wodehouse",
    "Ricardo Piglia/Amanda Hopkinson": "Ricardo Piglia",
    "Erin St. Claire/Sandra Brown": "Sandra Brown",
    "Rachel Ryan/Sandra Brown": "Sandra Brown",
    "Sandra Brown/Delia Mateovich": "Sandra Brown",
    "Terry Pratchett/Melvyn Grant": "Terry Pratchett",
    "Terry Pratchett/Paul Kidby": "Terry Pratchett",

    # Stephen King
    "Stephen King/Bernie Wrightson": "Stephen King",
    "Stephen King/Bernie Wrightson/Michele Wrightson": "Stephen King",
    "Stephen King/Bettina Blanch Tyroller": "Stephen King",
    "Stephen King/Campbell Scott": "Stephen King",
    "Stephen King/David Palladini": "Stephen King",
    "Stephen King/David Purdham": "Stephen King",
    "Stephen King/Eduardo Goligorsky": "Stephen King",
    "Stephen King/Glenn Chadbourne": "Stephen King",
    "Stephen King/Gregorio Vlastelica": "Stephen King",
    "Stephen King/Gregorio Vlastelica/Eduardo Goligorsky": "Stephen King",
    "Stephen King/Jerry N. Uelsmann": "Stephen King",
    "Stephen King/Joachim Körber": "Stephen King",
    "Stephen King/John  Glover": "Stephen King",
    "Stephen King/Josh Hamilton": "Stephen King",
    "Stephen King/Marie Milpois": "Stephen King",
    "Stephen King/Marta I. Guastavino": "Stephen King",
    "Stephen King/María Eugenia Ciocchini Suárez": "Stephen King",
    "Stephen King/María Mir": "Stephen King",
    "Stephen King/Ned Dameron": "Stephen King",
    "Stephen King/Richard Bachman": "Stephen King",
    "Stephen King/Ron McLarty": "Stephen King",
    "Stephen King/Verónica Canales": "Stephen King",
    "Stephen King/William Olivier Desmond/Maria Teresa Marenco": "Stephen King",
    "Richard Bachman/Stephen King": "Stephen King",
    "Richard Bachman/Stephen King/Hernán Sabaté Vargas": "Stephen King",
    "Richard Bachman/Stephen King/Joseph M. Apfelbäume": "Stephen King",

    # Keep both authors when they're colaborators
    "Stephen King/Peter Straub/Bernard Cohen": "Stephen King/Peter Straub",
    "Stephen King/Peter Straub/Pilar Giralt Gorina": "Stephen King/Peter Straub",
    "James Patterson/Andrew Gross/Carolyn McCormick": "James Patterson/Andrew Gross",
    "James Patterson/Melissa Leo/Jeremy Piven/Andrew Gross": "James Patterson/Andrew Gross",
    "Stephen Hawking/Leonard Mlodinow/David Jou i Mirabent": "Stephen Hawking/Leonard Mlodinow",
}

# Apply the corrections and show the 20 most common author credits
df_clean["authors"] = df_clean["authors"].replace(more_author_corrections)
df_clean["authors"].value_counts().head(20)

authors
Stephen King                   80
Sandra Brown                   46
Agatha Christie                45
P.G. Wodehouse                 43
Rumiko Takahashi               39
Orson Scott Card               37
Gabriel García Márquez         36
Piers Anthony                  30
Mercedes Lackey                29
Dick Francis                   28
Dan Simmons                    27
Terry Pratchett                26
J.K. Rowling                   25
J.R.R. Tolkien                 25
Janet Evanovich                25
Bill Bryson                    24
James Patterson                24
Margaret Weis/Tracy Hickman    23
Gordon Korman                  23
Laurell K. Hamilton            23
Name: count, dtype: int64

In [41]:
# Grouped spelling variants and selected editions with translator or other contributor credits under the same author name to simplify.
# We kept co-author credits and ambiguous cases separate. This lets us count editions and compare ratings more consistently by author.

# After standardising author credits, the top three changed from Stephen King with 40 editions, P.G. Wodehouse with 40 and
# Rumiko Takahashi with 39 to Stephen King npw with 80 editions, followed by Sandra Brown with 46 and Agatha Christie with 45.
# These counts show which authors are most represented in the catalogue.

In [42]:
#Some entries had "NOT A BOOK" as author
not_a_book = df_clean["authors"] == "NOT A BOOK"

print("Records with this label:", not_a_book.sum())
df_clean[not_a_book][["bookID", "title", "authors", "publisher"]]

Records with this label: 5


,bookID,title,authors,publisher
1061,3593,Murder by Moonlight & Other Mysteries (New Adv...,NOT A BOOK,Simon Schuster Audio
1064,3599,The Unfortunate Tobacconist & Other Mysteries ...,NOT A BOOK,Simon & Schuster Audio
5472,19786,The Goon Show Volume 4: My Knees Have Fallen ...,NOT A BOOK,BBC Physical Audio
5473,19787,The Goon Show: Moriarty Where Are You?,NOT A BOOK,BBC Physical Audio
5474,19788,The Goon Show Volume 11: He's Fallen in the W...,NOT A BOOK,BBC Physical Audio


In [43]:
#Turning the 5 entries as missing values so "NOT A BOOK" doesn't show up as an author
df_clean["authors"] = df_clean["authors"].replace("NOT A BOOK", float("nan"))

print("Remaining placeholders:", (df_clean["authors"] == "NOT A BOOK").sum())
print("Missing authors:", df_clean["authors"].isna().sum())

Remaining placeholders: 0
Missing authors: 5


In [44]:
#checking the dates
df_clean["publication_date"].head(10)

0    9/16/2006
1     9/1/2004
2    11/1/2003
3     5/1/2004
4    9/13/2004
5    4/26/2005
6    9/12/2005
7    11/1/2005
8    4/30/2002
9     8/3/2004
Name: publication_date, dtype: str

In [45]:
#isolating the year into a new column
df_clean["publication_year"] = df_clean["publication_date"].str.split("/").str[2]

df_clean[["publication_date", "publication_year"]].head(10)

,publication_date,publication_year
0,9/16/2006,2006
1,9/1/2004,2004
2,11/1/2003,2003
3,5/1/2004,2004
4,9/13/2004,2004
5,4/26/2005,2005
6,9/12/2005,2005
7,11/1/2005,2005
8,4/30/2002,2002
9,8/3/2004,2004


In [46]:
#checking for empty values
dates = df_clean["publication_date"]
years = df_clean["publication_year"]

print("Missing dates:", dates.isna().sum())
print("Blank dates:", (dates.str.strip() == "").sum())
print("Missing years:", years.isna().sum())
print("Blank years:", (years.str.strip() == "").sum())

Missing dates: 0
Blank dates: 0
Missing years: 0
Blank years: 0


In [47]:
df_clean["publication_year"] = df_clean["publication_year"].astype(int)

In [48]:
df_clean.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,publication_year
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling,4.57,0439785960,9780439785969,eng,652.0,2095690,27591,9/16/2006,Scholastic Inc.,2006
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling,4.49,0439358078,9780439358071,eng,870.0,2153167,29221,9/1/2004,Scholastic Inc.,2004
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352.0,6333,244,11/1/2003,Scholastic,2003
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling,4.56,043965548X,9780439655484,eng,435.0,2339585,36325,5/1/2004,Scholastic Inc.,2004
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling,4.78,0439682584,9780439682589,eng,2690.0,41428,164,9/13/2004,Scholastic,2004


In [50]:
df_clean.to_csv("../data/clean/bookworms_cleaned_data_file.csv", index=False, encoding="utf-8")